# **AI Piano Transcriptor & Typesetting Pipeline**

This interactive notebook allows you to upload any piano solo audio recording (WAV, MP3, etc.), transcribe it into MIDI using deep learning, warp its rubato timings to a steady beat grid, apply adaptive quantization to resolve triplets, statefully split it between hands, and typeset it into beautiful sheet music using **LilyPond**.

### **How to Use:**
1. Run the **Setup** cell to install LilyPond and Python dependencies.
2. Run the **Inference Settings** cell, upload your audio file, and set the tempo and key signature.
3. Run the **Typeset & Play** cell to view the sheet music and listen to the synthesized MIDI.

## **1. Setup Environment**
Install LilyPond, FluidSynth (for audio playback), and python dependencies, then clone the pipeline repository.

In [ ]:
#@title Install Dependencies (takes ~1-2 minutes) { display-mode: "form" }
import os
import sys

print("Installing system packages (LilyPond, FluidSynth, Poppler)...")
!apt-get update -qq
!apt-get install -y -qq lilypond fluidsynth poppler-utils &>/dev/null

print("Installing Python packages...")
!pip install -q pretty_midi librosa torch piano-transcription-inference soundfile scipy mido pdf2image

print("Cloning pipeline repository...")
!git clone https://github.com/rfarnham/ai_piano_transcriptor.git
sys.path.append('/content/ai_piano_transcriptor')

print("Setup complete!")

## **2. Inference Settings & File Upload**
Run this cell to set the key signature, title, composer, and upload your audio file.

In [ ]:
#@title Configuration & Upload { display-mode: "form" }
from google.colab import files

Title = "My Transcription" #@param {type:"string"}
Composer = "AI Transcriptor" #@param {type:"string"}
BPM = 112 #@param {type:"integer"}
Key_Flats = 0 #@param {type:"slider", min:0, max:7, step:1}
Key_Sharps = 0 #@param {type:"slider", min:0, max:7, step:1}

print("Please upload your piano audio file (.wav, .mp3, .m4a)...")
uploaded = files.upload()
if not uploaded:
    print("Error: No file uploaded.")
else:
    audio_filename = list(uploaded.keys())[0]
    print(f"Uploaded successfully: {audio_filename}")

## **3. Run Pipeline**
Execute the transcription and typesetting pipeline.

In [ ]:
# Change directory to the repository to run main.py
%cd /content/ai_piano_transcriptor

import subprocess

# Build the command
cmd = [
    "python3", "main.py",
    "--input-audio", f"/content/{audio_filename}",
    "--output-ly", "transcription.ly",
    "--bpm", str(BPM),
    "--key-flats", str(Key_Flats),
    "--key-sharps", str(Key_Sharps),
    "--title", Title,
    "--composer", Composer
]

print("Running transcription and typesetting...")
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print("Error running pipeline:")
    print(res.stderr)
else:
    print("Compiling LilyPond to PDF and MIDI...")
    compile_res = subprocess.run(["lilypond", "transcription.ly"], capture_output=True, text=True)
    print(compile_res.stdout)
    if compile_res.returncode == 0:
        print("Successfully compiled score!")
    else:
        print("LilyPond compilation warning/error:")
        print(compile_res.stderr)

## **4. View Sheet Music & Play Audio**
Render page 1 of the engraved score as an image and play the synthesized audio file.

In [ ]:
from IPython.display import Image, Audio, display
from pdf2image import convert_from_path

pdf_path = "transcription.pdf"
wav_path = "warped_synth.wav"

# 1. Display first page score preview
if os.path.exists(pdf_path):
    pages = convert_from_path(pdf_path, first_page=1, last_page=1)
    if pages:
        preview_img = "first_page.png"
        pages[0].save(preview_img, "PNG")
        print("\n--- Score Preview (Page 1) ---")
        display(Image(preview_img, width=800))

# 2. Audio playback of synthesized MIDI
if os.path.exists(wav_path):
    print("\n--- Play Synthesized Transcription ---")
    display(Audio(wav_path))

## **5. Download Outputs**
Download the final typeset PDF, MIDI, and WAV files.

In [ ]:
from google.colab import files

print("Preparing download files...")
for f in ["transcription.pdf", "transcription.midi", "warped_synth.wav"]:
    if os.path.exists(f):
        files.download(f)